# Revisão de Resultados — Intraday Regression Bands

Notebook para análise dos resultados dos backtests e walk-forward.
Execute os scripts 04 e 05 antes de usar este notebook.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.config import load_config, get_project_root

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

cfg = load_config()
root = get_project_root()

tables_dir = root / 'results' / 'tables'
figures_dir = root / 'results' / 'figures'
backtests_dir = root / 'data' / 'backtests'

print('Projeto:', root)
print('Tables dir exists:', tables_dir.exists())
print('Backtests dir exists:', backtests_dir.exists())

## 1. Sumário de Performance

In [ ]:
metrics_path = tables_dir / 'performance_summary.csv'

if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)
    print(f'Total de combinações: {len(df_metrics)}')
    print(f'Colunas: {list(df_metrics.columns)}')
    df_metrics.head(5)
else:
    print('Arquivo não encontrado. Execute: python scripts/04_run_backtests.py')
    df_metrics = pd.DataFrame()

## 2. Top estratégias por Sharpe (sem custos)

In [ ]:
if not df_metrics.empty:
    no_cost = df_metrics[(df_metrics['fee_bps'] == 0) & (df_metrics['slippage_bps'] == 0)].copy()
    top = (
        no_cost
        .sort_values('sharpe', ascending=False)
        .head(15)
        [['strategy', 'symbol', 'interval', 'sharpe', 'annualized_return',
          'annualized_volatility', 'max_drawdown', 'number_of_trades', 'hit_rate']]
    )
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(top.style.background_gradient(subset=['sharpe'], cmap='RdYlGn'))

## 3. Impacto dos custos no Sharpe

In [ ]:
if not df_metrics.empty and 'strategy' in df_metrics.columns:
    strategies = df_metrics['strategy'].unique()
    
    # Para cada estratégia, mostra Sharpe vs total custo (fee + slip)
    df_metrics['total_cost_bps'] = df_metrics['fee_bps'] + df_metrics['slippage_bps']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.tab10.colors
    
    for i, strat in enumerate(strategies):
        sub = df_metrics[df_metrics['strategy'] == strat].groupby('total_cost_bps')['sharpe'].mean().reset_index()
        ax.plot(sub['total_cost_bps'], sub['sharpe'], marker='o', markersize=6,
                label=strat, color=colors[i % len(colors)], linewidth=1.5)
    
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Custo total (fee + slippage, bps por lado)')
    ax.set_ylabel('Sharpe médio')
    ax.set_title('Sensibilidade do Sharpe ao Custo de Transação', fontweight='bold')
    ax.legend(fontsize=9, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()

## 4. Comparação com estratégia aleatória

In [ ]:
if not df_metrics.empty and 'strategy' in df_metrics.columns:
    cost_filter = (df_metrics['fee_bps'] == 2) & (df_metrics['slippage_bps'] == 2)
    sub = df_metrics[cost_filter].copy()
    
    comparison = (
        sub
        .groupby('strategy')[['sharpe', 'annualized_return', 'max_drawdown', 'number_of_trades']]
        .mean()
        .sort_values('sharpe', ascending=False)
    )
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bar_colors = ['#e74c3c' if idx == 'random' else '#3498db' for idx in comparison.index]
    bars = ax.barh(comparison.index, comparison['sharpe'], color=bar_colors, alpha=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    
    for bar, val in zip(bars, comparison['sharpe']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    
    ax.set_xlabel('Sharpe Médio (fee=2bps, slip=2bps)')
    ax.set_title('Sharpe por Estratégia vs Aleatória', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()
    
    print('\nComparação detalhada:')
    display(comparison)

## 5. Curvas de Equity (fee=2bps, slip=2bps)

In [ ]:
fee_tag = 'fee2_slip2'
equity_files = list(backtests_dir.glob(f'equity_*_{fee_tag}.parquet'))

if equity_files:
    fig, ax = plt.subplots(figsize=(14, 7))
    colors = plt.cm.tab10.colors
    
    for i, f in enumerate(equity_files[:10]):
        df_eq = pd.read_parquet(f)
        if 'timestamp' in df_eq.columns and 'equity' in df_eq.columns:
            eq = df_eq.set_index('timestamp')['equity']
            label = f.stem.replace(f'_{fee_tag}', '').replace('equity_', '')
            ax.plot(eq.index, eq / eq.iloc[0], label=label,
                    color=colors[i % len(colors)], linewidth=1.0)
    
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Equity Curves (fee=2bps, slip=2bps)', fontweight='bold')
    ax.set_ylabel('Retorno Acumulado (base=1)')
    ax.set_xlabel('Data')
    ax.legend(fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()
else:
    print('Nenhum arquivo de equity encontrado. Execute: python scripts/04_run_backtests.py')

## 6. Resultados Walk-forward

In [ ]:
wf_path = tables_dir / 'walk_forward_results.csv'
params_path = tables_dir / 'best_parameters_by_window.csv'

if wf_path.exists():
    df_wf = pd.read_csv(wf_path)
    print(f'Janelas walk-forward: {len(df_wf)}')
    
    if 'strategy' in df_wf.columns and 'sharpe' in df_wf.columns:
        wf_summary = (
            df_wf
            .groupby('strategy')[['sharpe', 'annualized_return', 'max_drawdown']]
            .agg(['mean', 'std'])
        )
        print('\nResumo Walk-forward (out-of-sample):')
        display(wf_summary)
else:
    print('Walk-forward não executado ainda. Execute: python scripts/05_run_walk_forward.py')

## 7. Break-even costs

In [ ]:
if not df_metrics.empty and 'break_even_cost_bps' in df_metrics.columns:
    no_cost = df_metrics[(df_metrics['fee_bps'] == 0) & (df_metrics['slippage_bps'] == 0)].copy()
    no_cost = no_cost.dropna(subset=['break_even_cost_bps'])
    
    bec = (
        no_cost
        .groupby(['strategy', 'symbol', 'interval'])['break_even_cost_bps']
        .mean()
        .reset_index()
        .sort_values('break_even_cost_bps', ascending=False)
    )
    
    labels = bec['strategy'] + ' | ' + bec['symbol'] + ' ' + bec['interval']
    colors = ['#27ae60' if v > 5 else '#e67e22' if v > 2 else '#e74c3c'
              for v in bec['break_even_cost_bps']]
    
    fig, ax = plt.subplots(figsize=(12, max(5, len(bec) * 0.4)))
    ax.barh(labels, bec['break_even_cost_bps'], color=colors, alpha=0.8)
    ax.axvline(2.0, color='orange', linewidth=1.5, linestyle='--', label='Custo típico (2 bps)')
    ax.axvline(5.0, color='red', linewidth=1.5, linestyle='--', label='Custo alto (5 bps)')
    ax.set_xlabel('Break-even cost (bps por lado)')
    ax.set_title('Custo de Break-even por Estratégia', fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()